# W07｜Docker / Compose / MCP｜教学版

**建议学习时间：4h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射完成 Docker + MCP | 只学当前项目必需部分 | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学 MCP Client / Server / Tool | 能解释 MCP vs REST/普通 Tool | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 写 Docker Compose | FastAPI + PostgreSQL 可启动 | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 一键启动核心系统 + 做 1 个最小 MCP Tool | compose up 后 API 可用 | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 画 MCP 调用链 | 能解释协议解决的是什么问题 | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜Docker Compose 参考
```yaml
services:
  api:
    build: .
    ports: ["8000:8000"]
    environment:
      DATABASE_URL: postgresql://user:pass@db:5432/chatbi
    depends_on: [db]
  db:
    image: postgres:16
    environment:
      POSTGRES_USER: user
      POSTGRES_PASSWORD: pass
      POSTGRES_DB: chatbi
```

### Task 4｜MCP Tool 参考
Tool 只接收 `metric/dimension/filters`，不要暴露 raw SQL。

### Task 5｜调用链
`User → Agent/Host → MCP Client → MCP Server → Safe Tool → Semantic Layer/DB`

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 理解容器基础
- 能写最小 Dockerfile/Compose
- 理解 MCP 的角色与安全边界

        ## 本周前置检查

        - 项目已能本地启动 FastAPI + DB

        ## 本周必须记住的英文术语

        - `image`：镜像
- `container`：容器
- `volume`：数据卷
- `network`：容器网络
- `MCP server`：MCP服务端
- `tool schema`：工具结构

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. Docker 解决什么问题？

    **是什么（What）**  
    把应用及运行依赖打包成可重复启动的 Image/Container。

    **为什么需要（Why）**  
    减少“我电脑能跑、服务器不能跑”的环境差异。

    **什么时候用（When）**  
    API/DB/Redis/Worker 的本地与部署环境。

    **常见误区 / 容易踩坑**  
    Docker 不等于 Kubernetes；Container 也不是虚拟机。


**中英术语**：`image` = 镜像；`container` = 容器；`volume` = 数据卷；`network` = 网络；`Docker Compose` = 多容器编排

### 代码实验：生成一个最小 Dockerfile

**这段代码想让你看什么？**

观察每条指令是在构建 image 还是定义 container 启动行为。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
from pathlib import Path
p = Path("w07_generated")
p.mkdir(exist_ok=True)

dockerfile = '''FROM python:3.12-slim
WORKDIR /app
COPY . .
RUN pip install -r requirements.txt
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''
(p / "Dockerfile").write_text(dockerfile, encoding="utf-8")
print(dockerfile)

## 2. MCP 是什么，不是什么？

    **是什么（What）**  
    MCP 是把 Tool/Resource 等能力以统一协议暴露给 AI Host/Client 的方式。

    **为什么需要（Why）**  
    减少每个 Agent 与外部系统之间的自定义集成。

    **什么时候用（When）**  
    企业内部工具、知识资源、数据访问能力。

    **常见误区 / 容易踩坑**  
    MCP 不是 Agent 框架，也不会自动让模型更聪明；Tool 权限和安全仍由你负责。


**中英术语**：`MCP server` = MCP服务端；`MCP client` = MCP客户端；`tool` = 工具；`resource` = 资源

### 代码实验：看懂 MCP Tool Schema

**这段代码想让你看什么？**

先理解协议契约，不急着运行真实 server。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
tool_schema = {
    "name": "run_analytics_query",
    "description": "Execute a validated analytics query plan",
    "inputSchema": {
        "type": "object",
        "properties": {
            "metric": {"type": "string"},
            "dimension": {"type": "string"},
        },
        "required": ["metric", "dimension"]
    }
}
print(tool_schema)

### Checkpoint

1. API container 里为什么不能用 `localhost` 访问另一个 DB container？
2. MCP 和 REST API 是竞争关系吗？
3. 为什么不能暴露 `execute_any_sql` tool？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 容器有独立网络命名空间；`localhost` 指自己。Compose 中通常通过 service name 访问其他容器。

**2.** 不是。MCP 是 AI 工具/资源集成协议，底层实现仍可能调用 HTTP/REST。

**3.** 它绕过 semantic/permission guardrail，模型一旦误调用或被 prompt injection 诱导会产生严重风险。

    </details>

### 练习：设计安全 MCP Tool

为 ChatBI 设计一个只接收 `metric + dimension + filters` 的 tool schema，禁止原始 SQL。

<details>
    <summary><b>参考答案</b></summary>

    ```python
tool = {
  "name": "analytics_query",
  "inputSchema": {
    "type": "object",
    "properties": {
      "metric": {"type": "string"},
      "dimension": {"type": "string"},
      "filters": {"type": "object"}
    },
    "required": ["metric", "dimension"]
  }
}
```

    </details>

## ChatBI 本周作业

        用 Docker Compose 启动 `api + postgres`。  
另外为安全 QueryPlan 设计一个 MCP Tool（先不要求全量 MCP 化）。

### 本周面试 / 理解检查

1. Docker Image 与 Container 区别？
2. Volume 为什么重要？
3. MCP 的价值是什么？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** Image 是只读模板；Container 是 image 的运行实例并带运行时状态。

**2.** 容器可以销毁重建，持久数据不能跟着消失，volume 用于持久化/挂载。

**3.** 给 AI Host 提供统一工具/资源发现与调用协议，降低集成碎片化。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。